📌 Module 1 — Product Performance Summary

Business Requirement

Create one table.

Expected output:

Product ID	Product Name	Category	Sub Category	Sales	Profit	Quantity	Profit Margin

You'll need to calculate:

Total Sales  
Total Profit  
Quantity Sold  
Profit Margin %

In [0]:
%sql
select `Product ID`, `Product Name`, `Category` , `Sub-Category`, round(sum(`Sales`),2) as Total_Sales, round(sum(`Profit`),2) as Total_Profit, sum(`Quantity`) as Total_Quantity, round((sum(`Profit`) / sum(`Sales`) )*100, 2) as Profit_Margin,
rank() over(order by sum(`Sales`) desc) as TotalSales_Rank,
rank() over(order by sum(`Profit`) desc) as TotalProfit_Rank,
rank() over(order by (sum(`Profit`) / sum(`Sales`) )*100 desc) as TotalMarginProfit_Rank
from abc_retail_pvt_ltd.data_analytics.superstore
group by `Product ID`, `Product Name`, `Category`, `Sub-Category`
ORDER BY Total_Sales DESC;

📌 Module 2 — Product Ranking

Rank products by

Revenue  
Profit  
Quantity Sold

In [0]:
%sql
SELECT `Product ID`, `Product Name`, ROUND(SUM(`Sales`),2) AS Total_Sales, ROUND(SUM(`Profit`),2) AS Total_Profit,
SUM(`Quantity`) AS Total_Quantity,
RANK() OVER(ORDER BY SUM(`Sales`) DESC) AS Sales_Rank,
RANK() OVER(ORDER BY SUM(`Profit`) DESC) AS Profit_Rank,
RANK() OVER(ORDER BY SUM(`Quantity`) DESC) AS Quantity_Rank
FROM abc_retail_pvt_ltd.data_analytics.superstore
GROUP BY `Product ID`, `Product Name`;

📌 Module 3 — Category Performance

Business asks

Which category contributes the most revenue?

Analyze

Furniture  
Office Supplies  
Technology

In [0]:
%sql
SELECT Category, ROUND(SUM(`Sales`), 2) AS Total_Sales,
ROUND(SUM(`Profit`),2) AS Total_Profit,
SUM(`Quantity`) AS Quantity,
ROUND((SUM(`Profit`)/SUM(`Sales`))*100,2) AS Profit_Margin,
rank() over(order by sum(`Sales`) desc) as TotalSales_Rank
FROM abc_retail_pvt_ltd.data_analytics.superstore
GROUP BY Category

Finding

Technology generated the highest revenue of $839,893, making it the company's best-performing category.

Business Insight

Customers have higher demand for technology products compared to other categories.

Recommendation

Increase inventory and marketing investments for Technology while continuing to monitor profitability.

📌 Module 4 — Sub-Category Performance

Now drill deeper.

Instead of Category,

analyze

Chairs
Phones
Tables
Binders
Storage

etc.

In [0]:
%sql
SELECT`Sub-Category`, ROUND(SUM(`Sales`),2) AS Total_Sales, ROUND(SUM(`Profit`),2) AS Total_Profit, SUM(`Quantity`) AS Quantity,
ROUND((SUM(`Profit`)/SUM(`Sales`))*100,2) AS Profit_Margin,
rank() over(order by sum(`Sales`) desc) as TotalSales_Rank
FROM abc_retail_pvt_ltd.data_analytics.superstore
GROUP BY `Sub-Category`

📌 Module 5 — High Revenue, Low Profit Products

One of my favorite analyses.

Business Question

Which products sell a lot but barely make money?

Why?

Maybe

discounts are too high
shipping costs are high
product cost is high

These products need investigation.

In [0]:
%sql
WITH product_performance AS 
(
SELECT `Product ID`, `Product Name`, ROUND(SUM(Sales), 2) AS Total_Sales, ROUND(SUM(Profit), 2) AS Total_Profit,
RANK() OVER (ORDER BY SUM(Sales) DESC) AS Sales_Rank,
RANK() OVER (ORDER BY SUM(Profit) ASC) AS Profit_Rank,
ROUND((SUM(`Profit`)/SUM(`Sales`))*100,2) AS Profit_Margin
FROM abc_retail_pvt_ltd.data_analytics.superstore
GROUP BY `Product ID`,`Product Name`
)

SELECT * FROM product_performance
WHERE Total_Sales>10000
ORDER BY Profit_Margin ASC;

📌 Module 6 — Loss Making Products

Business asks

Which products are losing money?

Expected output

Product	Sales	Profit

Sort by

Lowest Profit.

In [0]:
%sql
WITH loss_products AS 
(
SELECT `Product ID`, `Product Name`, ROUND(SUM(Sales), 2) AS Total_Sales, ROUND(SUM(Profit), 2) AS Total_Profit,
ROUND((SUM(`Profit`)/SUM(`Sales`))*100,2) AS Profit_Margin,
DENSE_RANK() OVER (ORDER BY SUM(Profit) ASC) AS Loss_Rank
FROM abc_retail_pvt_ltd.data_analytics.superstore
GROUP BY `Product ID`, `Product Name`
)

SELECT *
FROM loss_products
WHERE Total_Profit < 0
ORDER BY Loss_Rank;

📌 Module 7 — Product Segmentation

Similar to customers.

Example

Sales > 15000

↓

Best Seller

Sales between 8000-15000

↓

High Performer

Sales between 3000-8000

↓

Average Performer

Below 3000

↓

Low Performer

In [0]:
%sql
SELECT `Product Name`, ROUND(SUM(Sales), 2) AS Total_Sales,
CASE
WHEN SUM(Sales) > 15000 THEN 'Best Seller'
WHEN SUM(Sales) BETWEEN 8000 AND 15000 THEN 'High Performer'
WHEN SUM(Sales) BETWEEN 3000 AND 8000 THEN 'Average Performer'
ELSE 'Low Performer'
END AS Product_Segment
FROM abc_retail_pvt_ltd.data_analytics.superstore
GROUP BY `Product Name`
ORDER BY Total_Sales DESC;

In [0]:
%sql
WITH product_segment AS 
(
SELECT `Product Name`, SUM(Sales) AS Total_Sales,
CASE
WHEN SUM(Sales) > 15000 THEN 'Best Seller'
WHEN SUM(Sales) BETWEEN 8000 AND 15000 THEN 'High Performer'
WHEN SUM(Sales) BETWEEN 3000 AND 8000 THEN 'Average Performer'
ELSE 'Low Performer'
END AS Product_Segment
FROM abc_retail_pvt_ltd.data_analytics.superstore
GROUP BY `Product Name`
)

SELECT Product_Segment,
COUNT(*) AS No_of_Products
FROM product_segment
GROUP BY Product_Segment
ORDER BY No_of_Products DESC;

📌 Module 8 — Executive Insights

After every query

write

Finding

↓

Business Insight

↓

Recommendation

## Finding 1

Technology products generate the highest revenue among all categories.

Business Insight

Technology products have strong customer demand and contribute significantly to overall sales.

Recommendation

Increase inventory and marketing investment in Technology products while monitoring stock availability.

## Finding 2

Some products generate high revenue but low profit margins.

Business Insight

High discounts or procurement costs may be reducing profitability despite strong sales.

Recommendation

Review pricing strategy and supplier costs for these products.

## Finding 3

Several products consistently generate negative profits.

Business Insight

These products reduce overall company profitability.

Recommendation

Analyze the root cause, renegotiate supplier contracts, revise pricing, or discontinue products with persistent losses.

## Finding 4

A small number of products contribute a large share of total revenue.

Business Insight

Revenue is concentrated in a limited set of products, making them strategically important.

Recommendation

Ensure consistent stock availability, monitor demand, and prioritize these products in marketing campaigns.